# EDL Full Experiment (auto-parallel)

10-fold cross-validation comparison of 8 LDL models across 3 datasets
(`SJAFFE`, `SBU_3DFE`, `Human_Gene`).

The pool config is **auto-detected** by `edl_workers.auto_pool_config()`:

- **GPU mode** if ≥ 2 CUDA GPUs are visible to `nvidia-smi` — one worker
  pinned per GPU.
- **CPU mode** otherwise — a small number of fat CPU workers (≈
  `cpu_count // 4`) so TF's intra/inter-op threads don't oversubscribe.

The actual training loop lives in `edl_workers.py` next to this notebook
so loky subprocesses can `import edl_workers` cleanly. TensorFlow is
imported lazily inside each worker after `CUDA_VISIBLE_DEVICES` is pinned.

For each (model, dataset) pair we record six distributional metrics
(`chebyshev`, `clark`, `canberra`, `kl_divergence`, `cosine`, `intersection`)
across 10 folds with a 10% test split per fold, then summarise as
mean ± std.

For the evidential models (`EDL_LDL`, `BEDL_LDL`) and `SNEFY_LDL` we also
report:

- **Mean uncertainty** — average per-sample uncertainty on the test set
  (lower = the model is more confident).
- **Uncertainty calibration (Spearman ρ)** — rank correlation between
  per-sample uncertainty and per-sample KL divergence error.
  Higher = uncertainty tracks error better.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from loky import get_reusable_executor

# Make edl_workers importable whether the kernel started in `demo/` or in the
# project root.
_demo_dir = Path.cwd() if (Path.cwd() / 'edl_workers.py').exists() else Path.cwd() / 'demo'
if str(_demo_dir) not in sys.path:
    sys.path.insert(0, str(_demo_dir))

from pyldl.utils import load_dataset
from edl_workers import (
    init_worker, run_one_fold, auto_pool_config,
    MODEL_NAMES, METRICS,
)


E0000 00:00:1777062077.517584 3498275 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777062078.288646 3498275 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777062082.938172 3498275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777062082.938220 3498275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777062082.938222 3498275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777062082.938224 3498275 computation_placer.cc:177] computation placer already registered. Please check linka

## Configuration

`auto_pool_config()` picks GPU vs CPU mode based on what's actually present.
Override `MODE_OVERRIDE` if you want to force one or the other (useful for
A/B testing GPU-vs-CPU on the same box).


In [2]:
DATASETS = ['SJAFFE', 'SBU_3DFE', 'Movie', 'Music']
N_SPLITS = 10
N_EPOCHS = 100
RANDOM_STATE = 0

# --- Parallel config (auto-detected) -------------------------------------
# Set to None to auto-detect, or pass a dict to override e.g.
#   POOL_CFG = {'mode': 'CPU', 'gpu_ids': [], 'n_workers': 4,
#               'intra_op_threads': 1, 'inter_op_threads': 1}
POOL_CFG = None
# -------------------------------------------------------------------------

if POOL_CFG is None:
    POOL_CFG = auto_pool_config()

GPU_IDS   = POOL_CFG['gpu_ids']
N_WORKERS = POOL_CFG['n_workers']
INTRA     = POOL_CFG['intra_op_threads']
INTER     = POOL_CFG['inter_op_threads']
MODE      = POOL_CFG['mode']

print(f'mode      : {MODE}')
print(f'workers   : {N_WORKERS}')
print(f'gpu_ids   : {GPU_IDS if GPU_IDS else "(CPU only)"}')
print(f'tf threads: intra={INTRA}, inter={INTER}')


mode      : GPU
workers   : 3
gpu_ids   : [0, 1, 2]
tf threads: intra=1, inter=1


## Build the worker pool

Each worker pulls one entry off `gpu_queue` exactly once at startup
(loky's `initializer`) and pins `CUDA_VISIBLE_DEVICES` before TF sees a
GPU. In CPU mode the entry is `None`, which triggers
`tf.config.set_visible_devices([], 'GPU')` so TF can't accidentally find
the GPU through another path. `reuse=False` forces a fresh pool if you
re-run this cell after changing the config.


In [7]:
mgr = mp.Manager()
gpu_queue = mgr.Queue()

slots = list(GPU_IDS) if GPU_IDS else [None] * N_WORKERS
assert len(slots) == N_WORKERS, 'one queue slot per worker'
for g in slots:
    gpu_queue.put(g)

executor = get_reusable_executor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(gpu_queue, INTRA, INTER),
    reuse=False,
)
print(f'pool ready: {N_WORKERS} workers, slots={slots}')


pool ready: 3 workers, slots=[0, 1, 2]


## Sanity check: what do the workers actually see?

Submit one cheap probe per worker and confirm that:

- **GPU mode**: each worker reports a different `CUDA_VISIBLE_DEVICES`
  and a single GPU device.
- **CPU mode**: every worker reports `CUDA_VISIBLE_DEVICES = '-1'` and
  an empty `gpu_devices` list.

If a row shows the wrong state, the pool's pinning didn't take effect —
re-run the pool cell to get a fresh pool.


In [20]:
def _check():
    import os, tensorflow as tf
    return {
        'pid': os.getpid(),
        'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES'),
        'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
        'intra_threads': tf.config.threading.get_intra_op_parallelism_threads(),
        'inter_threads': tf.config.threading.get_inter_op_parallelism_threads(),
    }

# Submit several so loky has a reason to spin up every worker.
checks = [executor.submit(_check) for _ in range(N_WORKERS * 4)]
df = pd.DataFrame([c.result() for c in checks]).drop_duplicates(subset='pid').reset_index(drop=True)
print(f'{len(df)} unique workers (expected {N_WORKERS})')
df


E0000 00:00:1777057187.431898 1117820 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777057187.436662 1117820 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777057187.449214 1117820 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777057187.449312 1117820 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777057187.449343 1117820 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777057187.449368 1117820 computation_placer.cc:177] computation placer already registered. Please check linka

1 unique workers (expected 12)


,pid,CUDA_VISIBLE_DEVICES,gpu_devices,intra_threads,inter_threads
0,1117820,-1,[],1,1


/home/dcs01/.local/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/home/dcs01/.local/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Build the job list

`KFold` splits are generated in the parent (deterministic, cheap) and the
fold slices are passed to workers as numpy arrays. Loky memmaps large
numpy arrays automatically, so this is fast even for the bigger datasets.


In [6]:
DATASETS = ['SJAFFE', 'SBU_3DFE', 'Music', 'Flickr', 'Yeast_alpha']
jobs = []
for dataset_name in DATASETS:
    X, D = load_dataset(dataset_name, dir='dataset')
    print(f'{dataset_name}: {X.shape[0]} samples, {X.shape[1]} features, {D.shape[1]} labels')
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        Xtr, Xte = X[train_idx], X[test_idx]
        Dtr, Dte = D[train_idx], D[test_idx]
        for model_name in MODEL_NAMES:
            jobs.append((dataset_name, model_name, fold_idx, Xtr, Dtr, Xte, Dte))

total = len(jobs)
print(f'queued {total} jobs ({len(DATASETS)} datasets × {N_SPLITS} folds × {len(MODEL_NAMES)} models)')


SJAFFE: 213 samples, 243 features, 6 labels
SBU_3DFE: 2500 samples, 243 features, 6 labels
Music: 360 samples, 128 features, 9 labels
Flickr: 11150 samples, 168 features, 8 labels
Yeast_alpha: 2465 samples, 24 features, 18 labels
queued 400 jobs (5 datasets × 10 folds × 8 models)


## Submit + collect

Submission is non-blocking; results stream back via `as_completed` so the
log shows progress as folds finish. Per-fold failures are caught and
printed but don't stop the run.


In [ ]:
futures = {
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, N_EPOCHS): (ds, m, fi)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte) in jobs
}

raw_results = []
failures = []
for i, fut in enumerate(as_completed(futures), start=1):
    ds, m, fi = futures[fut]
    try:
        raw_results.append(fut.result())
        status = 'ok'
    except Exception as e:
        failures.append((ds, m, fi, repr(e)))
        status = f'FAILED ({type(e).__name__}: {e})'
    print(f'[{i:4d}/{total}] {ds:12s} | fold {fi:2d} | {m:30s} {status}')

print(f'\ndone: {len(raw_results)} ok, {len(failures)} failed')


E0000 00:00:1777062438.615079 3498904 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777062438.619191 3498904 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777062438.640659 3498904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777062438.640689 3498904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777062438.640691 3498904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777062438.640693 3498904 computation_placer.cc:177] computation placer already registered. Please check linka

[   1/400] SJAFFE       | fold  1 | BEDL_LDL (loglikelihood)       ok
[   2/400] SJAFFE       | fold  1 | EDL_LDL (loglikelihood)        ok
[   3/400] SJAFFE       | fold  1 | EDL_LDL (bayes_mse)            ok
[   4/400] SJAFFE       | fold  1 | BEDL_LDL (bayes_mse)           ok
[   5/400] SJAFFE       | fold  1 | AA_BP                          ok
[   6/400] SJAFFE       | fold  1 | SA_BFGS                        ok
[   7/400] SJAFFE       | fold  2 | EDL_LDL (loglikelihood)        ok
[   8/400] SJAFFE       | fold  1 | Duo_LDL                        ok
[   9/400] SJAFFE       | fold  2 | EDL_LDL (bayes_mse)            ok
[  10/400] SJAFFE       | fold  1 | SNEFY_LDL                      ok
[  11/400] SJAFFE       | fold  2 | BEDL_LDL (loglikelihood)       ok
[  12/400] SJAFFE       | fold  2 | BEDL_LDL (bayes_mse)           ok
[  13/400] SJAFFE       | fold  2 | AA_BP                          ok
[  14/400] SJAFFE       | fold  2 | SA_BFGS                        ok
[  15/400] SJAFFE   

In [ ]:
import os, psutil

# Pool's worker PIDs — loky stashes them on the executor.
worker_pids = list(executor._processes.keys())
alive = []
dead = []
for pid in worker_pids:
    try:
        p = psutil.Process(pid)
        if p.is_running() and p.status() != psutil.STATUS_ZOMBIE:
            alive.append((pid, p.status(), p.memory_info().rss / 1e9, p.cpu_percent(interval=0.5)))
        else:
            dead.append(pid)
    except psutil.NoSuchProcess:
        dead.append(pid)

print(f'alive workers ({len(alive)}/{N_WORKERS}):')
for pid, status, rss_gb, cpu in alive:
    print(f'  pid={pid}  status={status}  rss={rss_gb:.1f}GB  cpu={cpu:.0f}%')
print(f'dead workers: {dead}')

print(f'\\nresults so far: {len(raw_results)} / {total}')
print(f'completed futures: {sum(f.done() for f in futures)}')
print(f'pending futures:   {sum(not f.done() for f in futures)}')


## Bucket results into per-model DataFrames

`per_model_results[(dataset, model_name)]` is a DataFrame with one row per
fold; columns are the recorded metrics.


In [ ]:
buckets = defaultdict(list)
for r in raw_results:
    buckets[(r['dataset'], r['model'])].append(r['scores'])

per_model_results = {key: pd.DataFrame(rows) for key, rows in buckets.items()}
print(f'{len(per_model_results)} (dataset, model) combinations have results')


8 (dataset, model) combinations have results


## Per-model fold tables

Inspect any single (dataset, model) DataFrame:


In [23]:
# Pick the first (dataset, model) that actually has results so this cell
# doesn't KeyError if you ran with a reduced DATASETS list.
if per_model_results:
    first_key = next(iter(per_model_results))
    print(f'showing: {first_key}')
    display(per_model_results[first_key])
else:
    print('no results yet — run the submit cell first')


showing: ('SJAFFE', 'EDL_LDL (loglikelihood)')


,chebyshev,clark,canberra,kl_divergence,cosine,intersection,mean_uncertainty,uncertainty_calibration
0,0.122201,0.433039,0.904852,0.074845,0.929207,0.845599,0.857143,0.077099
1,0.118690,0.422912,0.880232,0.072771,0.931808,0.850219,0.857143,0.210789


## Combined summary — mean ± std across folds

One row per (dataset, model); columns are `metric_mean` / `metric_std`.


In [24]:
def summarize(df):
    out = {}
    for col in df.columns:
        out[f'{col}_mean'] = df[col].mean()
        out[f'{col}_std']  = df[col].std()
    return out


summary_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name, **summarize(df)}
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index(['dataset', 'model'])
summary


chebyshev_mean  chebyshev_std  clark_mean  \
dataset model                                                                 
SJAFFE  EDL_LDL (loglikelihood)         0.120446       0.002482    0.427975   
        EDL_LDL (bayes_mse)             0.120166       0.002755    0.428287   
        AA_BP                           0.219314       0.034540    0.731415   
        BEDL_LDL (bayes_mse)            0.120353       0.004065    0.428580   
        Duo_LDL                         0.138574       0.007033    0.502687   
        BEDL_LDL (loglikelihood)        0.120232       0.003591    0.427045   
        SNEFY_LDL                       0.499595       0.332622    1.595904   
        SA_BFGS                         0.098753       0.003929    0.366276   

                                  clark_std  canberra_mean  canberra_std  \
dataset model                                                              
SJAFFE  EDL_LDL (loglikelihood)    0.007161       0.892542      0.017409   
        EDL_LDL (bayes_mse)        0.006561       0.896028      0.006493   
        AA_BP                      0.178870       1.469661      0.374993   
        BEDL_LDL (bayes_mse)       0.012366       0.899404      0.024034   
        Duo_LDL                    0.040572       0.993668      0.086851   
        BEDL_LDL (loglikelihood)   0.012188       0.894914      0.026550   
        SNEFY_LDL                  0.424672       3.591615      1.352791   
        SA_BFGS                    0.010629       0.743068      0.016277   

                                  kl_divergence_mean  kl_divergence_std  \
dataset model                                                             
SJAFFE  EDL_LDL (loglikelihood)             0.073808           0.001467   
        EDL_LDL (bayes_mse)                 0.074180           0.001101   
        AA_BP                               0.250959           0.106250   
        BEDL_LDL (bayes_mse)                0.073799           0.002472   
        Duo_LDL                             0.095509           0.014501   
        BEDL_LDL (loglikelihood)            0.073476           0.002619   
        SNEFY_LDL                           1.319181           0.836447   
        SA_BFGS                             0.054899           0.003871   

                                  cosine_mean  cosine_std  intersection_mean  \
dataset model                                                                  
SJAFFE  EDL_LDL (loglikelihood)      0.930507    0.001839           0.847909   
        EDL_LDL (bayes_mse)          0.930144    0.001429           0.847272   
        AA_BP                        0.799937    0.043767           0.736480   
        BEDL_LDL (bayes_mse)         0.930433    0.002786           0.846830   
        Duo_LDL                      0.911324    0.010604           0.829768   
        BEDL_LDL (loglikelihood)     0.930746    0.002883           0.847564   
        SNEFY_LDL                    0.567506    0.205585           0.438371   
        SA_BFGS                      0.949125    0.004245           0.874709   

                                  intersection_std  mean_uncertainty_mean  \
dataset model                                                               
SJAFFE  EDL_LDL (loglikelihood)           0.003267               0.857143   
        EDL_LDL (bayes_mse)               0.001490               0.857143   
        AA_BP                             0.043676                    NaN   
        BEDL_LDL (bayes_mse)              0.004384               0.385211   
        Duo_LDL                           0.012834                    NaN   
        BEDL_LDL (loglikelihood)          0.004808               0.402671   
        SNEFY_LDL                         0.244894               0.257603   
        SA_BFGS                           0.003888                    NaN   

                                  mean_uncertainty_std  \
dataset model                                            
SJAFFE  EDL_LDL (loglikelihood)            

### Compact view: `mean ± std` per metric


In [14]:
def fmt(mean, std):
    if pd.isna(mean):
        return ''
    return f'{mean:.4f} ± {std:.4f}'


compact_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name}
    for col in df.columns:
        row[col] = fmt(df[col].mean(), df[col].std())
    compact_rows.append(row)

compact = pd.DataFrame(compact_rows).set_index(['dataset', 'model'])
compact


chebyshev            clark  \
dataset model                                                        
SJAFFE  EDL_LDL (loglikelihood)   0.1204 ± 0.0025  0.4280 ± 0.0072   
        EDL_LDL (bayes_mse)       0.1202 ± 0.0028  0.4283 ± 0.0066   
        AA_BP                     0.2193 ± 0.0345  0.7314 ± 0.1789   
        BEDL_LDL (bayes_mse)      0.1204 ± 0.0041  0.4286 ± 0.0124   
        Duo_LDL                   0.1386 ± 0.0070  0.5027 ± 0.0406   
        BEDL_LDL (loglikelihood)  0.1202 ± 0.0036  0.4270 ± 0.0122   
        SNEFY_LDL                 0.4996 ± 0.3326  1.5959 ± 0.4247   
        SA_BFGS                   0.0988 ± 0.0039  0.3663 ± 0.0106   

                                         canberra    kl_divergence  \
dataset model                                                        
SJAFFE  EDL_LDL (loglikelihood)   0.8925 ± 0.0174  0.0738 ± 0.0015   
        EDL_LDL (bayes_mse)       0.8960 ± 0.0065  0.0742 ± 0.0011   
        AA_BP                     1.4697 ± 0.3750  0.2510 ± 0.1062   
        BEDL_LDL (bayes_mse)      0.8994 ± 0.0240  0.0738 ± 0.0025   
        Duo_LDL                   0.9937 ± 0.0869  0.0955 ± 0.0145   
        BEDL_LDL (loglikelihood)  0.8949 ± 0.0265  0.0735 ± 0.0026   
        SNEFY_LDL                 3.5916 ± 1.3528  1.3192 ± 0.8364   
        SA_BFGS                   0.7431 ± 0.0163  0.0549 ± 0.0039   

                                           cosine     intersection  \
dataset model                                                        
SJAFFE  EDL_LDL (loglikelihood)   0.9305 ± 0.0018  0.8479 ± 0.0033   
        EDL_LDL (bayes_mse)       0.9301 ± 0.0014  0.8473 ± 0.0015   
        AA_BP                     0.7999 ± 0.0438  0.7365 ± 0.0437   
        BEDL_LDL (bayes_mse)      0.9304 ± 0.0028  0.8468 ± 0.0044   
        Duo_LDL                   0.9113 ± 0.0106  0.8298 ± 0.0128   
        BEDL_LDL (loglikelihood)  0.9307 ± 0.0029  0.8476 ± 0.0048   
        SNEFY_LDL                 0.5675 ± 0.2056  0.4384 ± 0.2449   
        SA_BFGS                   0.9491 ± 0.0042  0.8747 ± 0.0039   

                                 mean_uncertainty uncertainty_calibration  
dataset model                                                              
SJAFFE  EDL_LDL (loglikelihood)   0.8571 ± 0.0000         0.1439 ± 0.0945  
        EDL_LDL (bayes_mse)       0.8571 ± 0.0000        -0.0118 ± 0.0935  
        AA_BP                                 NaN                     NaN  
        BEDL_LDL (bayes_mse)      0.3852 ± 0.0011         0.1867 ± 0.2745  
        Duo_LDL                               NaN                     NaN  
        BEDL_LDL (loglikelihood)  0.4027 ± 0.0050        -0.1086 ± 0.1315  
        SNEFY_LDL                 0.2576 ± 0.1916        -0.0608 ± 0.2045  
        SA_BFGS                               NaN                     NaN

## Uncertainty results (EDL_LDL, BEDL_LDL, SNEFY_LDL only)

- `mean_uncertainty` — average per-sample uncertainty on test (model-specific
  scale; lower = more confident).
- `uncertainty_calibration` — Spearman ρ between per-sample uncertainty and
  per-sample KL divergence error. Higher = uncertainty better predicts error.


In [15]:
uncertainty_models = {
    'EDL_LDL (loglikelihood)', 'EDL_LDL (bayes_mse)',
    'BEDL_LDL (loglikelihood)', 'BEDL_LDL (bayes_mse)',
    'SNEFY_LDL',
}

uncertainty_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if model_name not in uncertainty_models or df.empty:
        continue
    if 'mean_uncertainty' not in df.columns:
        continue
    uncertainty_rows.append({
        'dataset': dataset_name,
        'model': model_name,
        'mean_uncertainty':        fmt(df['mean_uncertainty'].mean(),        df['mean_uncertainty'].std()),
        'uncertainty_calibration': fmt(df['uncertainty_calibration'].mean(), df['uncertainty_calibration'].std()),
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows).set_index(['dataset', 'model'])
uncertainty_summary


mean_uncertainty uncertainty_calibration
dataset model                                                            
SJAFFE  EDL_LDL (loglikelihood)   0.8571 ± 0.0000         0.1439 ± 0.0945
        EDL_LDL (bayes_mse)       0.8571 ± 0.0000        -0.0118 ± 0.0935
        BEDL_LDL (bayes_mse)      0.3852 ± 0.0011         0.1867 ± 0.2745
        BEDL_LDL (loglikelihood)  0.4027 ± 0.0050        -0.1086 ± 0.1315
        SNEFY_LDL                 0.2576 ± 0.1916        -0.0608 ± 0.2045

## Shut down the pool

Loky reuses pools by default; close it explicitly when you're done so the
worker processes (and any GPU memory they hold) are released.


In [16]:
executor.shutdown(wait=True, kill_workers=True)
